# Exploring `@apimodel`

`@apimodel` is a small decorator for turning annotated classes into API response models. It registers the class, injects a dict-based constructor, adds `from_dict` and `as_dict`, and supports lazy hydration with `Lazy[T]`.

This notebook treats the examples as little executable checks. Several cells use `assert` so you can rerun the notebook after changes and catch behavior shifts quickly.

## Setup

Keep the helpers tiny: one JSON loader, one cache-key inspector, and one compact project summary.

In [42]:
from __future__ import annotations

import copy
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Annotated

repo_root = Path.cwd()
if not (repo_root / "pypercache").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))

from pypercache.models.apimodel import Alias, Columns, Lazy, Timestamp, apimodel
from pypercache.models.validation import ApiModelValidationError
from pypercache.utils import UNSET

data_path = repo_root / "notebooks" / "apimodel_deep_sample.json"

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def lazy_cache_keys(obj) -> tuple[str, ...]:
    return tuple(sorted(k for k in vars(obj) if k.startswith("_lazycache_")))

def project_line(project) -> str:
    return f"{project.id}: {project.name} ({project.status})"

## A Small Model

The generated constructor accepts a raw dict. Annotated fields are hydrated, missing fields become `UNSET`, and extra raw keys remain available through `as_dict()`.

In [43]:
@apimodel
class TinyUser:
    id: int
    name: str
    missing_ok: str

tiny = TinyUser({"id": "7", "name": "Mina", "extra": "kept raw"})

assert tiny.id == 7
assert tiny.name == "Mina"
assert tiny.missing_ok is UNSET
assert tiny.as_dict()["extra"] == "kept raw"
tiny

TinyUser(id=7, name='Mina', missing_ok=UNSET)

`validate=True` checks the hydrated values against the annotations. Castable values are accepted after coercion, while values that cannot become the requested type raise `ApiModelValidationError`.

In [44]:
@apimodel(validate=True)
class CheckedUser:
    id: int
    name: str
    tags: list[str]

checked = CheckedUser({"id": "7", "name": "Mina", "tags": ["admin", "beta"]})

assert checked.id == 7
assert checked.tags == ["admin", "beta"]

try:
    CheckedUser({"id": "not an int", "name": "Mina", "tags": ["admin"]})
except ApiModelValidationError as error:
    validate_message = str(error)
else:
    raise AssertionError("validate=True should reject an uncastable id")

validate_message

"CheckedUser.id expected <class 'int'>, got str (not an int)"

`strict=True` is about presence, not truthiness. Missing annotated fields raise immediately, but normal falsy values like `0`, `False`, and `""` are still valid data.

In [45]:
@apimodel(strict=True)
class StrictSwitch:
    enabled: bool
    retries: int
    note: str

switch = StrictSwitch({"enabled": False, "retries": 0, "note": ""})

assert switch.enabled is False
assert switch.retries == 0
assert switch.note == ""

try:
    StrictSwitch({"enabled": True, "retries": 1})
except ApiModelValidationError as error:
    strict_message = str(error)
else:
    raise AssertionError("strict=True should reject a missing annotated field")

strict_message

'StrictSwitch.note is UNSET'

`validate` and `strict` can be combined when you want both guarantees: every annotated field must be present, and each hydrated value must match its annotation.

In [46]:
@apimodel(validate=True, strict=True)
class CompleteUser:
    id: int
    name: str

complete = CompleteUser({"id": "8", "name": "Bob"})

assert complete.id == 8
assert complete.name == "Bob"

try:
    CompleteUser({"id": "bad"})
except ApiModelValidationError as error:
    combined_message = str(error)
else:
    raise AssertionError("validate=True, strict=True should reject bad or incomplete data")

combined_message

"CompleteUser.id expected <class 'int'>, got str (bad)"

A lazy field is declared with `Lazy[T]`. The raw value stays in the original dict until the attribute is read, then it hydrates into the dependent model and is cached on that instance.

In [47]:
@apimodel
class TinyProfile:
    handle: str
    reputation: int
    
    def __post_init__(self):
        print('[TinyProfile has been hydrated]')

@apimodel
class TinyAccount:
    id: int
    profile: Lazy[TinyProfile]

account = TinyAccount({"id": 3, "profile": {"handle": "mina", "reputation": "42"}})

print(" -- Cache state -- ")
assert lazy_cache_keys(account) == ()

print("Before access: ", lazy_cache_keys(account))
profile = account.profile
print("After access: ", lazy_cache_keys(account))

assert isinstance(profile, TinyProfile)
assert profile.reputation == 42
assert account.profile is profile
assert lazy_cache_keys(account) == ("_lazycache_profile",)

profile.handle, lazy_cache_keys(account)

 -- Cache state -- 
Before access:  ()
[TinyProfile has been hydrated]
After access:  ('_lazycache_profile',)


('mina', ('_lazycache_profile',))

When `validate=True` is enabled, fields declared with `Lazy[T]` will be eagerly loaded because validation forces them to hydrated during construction.

In [48]:
@apimodel(validate=True)
class ValidatedLazyProfile:
    handle: str
    reputation: int

    def __post_init__(self):
        print('[ValidatedLazyProfile has been hydrated during __init__]')

@apimodel(validate=True)
class ValidatedLazyAccount:
    id: int
    profile: Lazy[ValidatedLazyProfile]

validated_account = ValidatedLazyAccount({
    "id": "3",
    "profile": {"handle": "mina", "reputation": "42"},
})

print(" -- Cache state -- ")
print("Before access: ", lazy_cache_keys(validated_account))
profile = validated_account.profile
print("After access: ", lazy_cache_keys(validated_account))

try:
    ValidatedLazyAccount({
        "id": 4,
        "profile": {"handle": "rowan", "reputation": "not a number"},
    })
except ApiModelValidationError as error:
    validation_message = str(error)
else:
    raise AssertionError("validate=True should reject bad lazy payloads during construction")

assert profile.reputation == 42
assert lazy_cache_keys(validated_account) == ("_lazycache_profile",)
assert "expected" in validation_message

profile.reputation, validation_message

[ValidatedLazyProfile has been hydrated during __init__]
 -- Cache state -- 
Before access:  ('_lazycache_profile',)
After access:  ('_lazycache_profile',)


(42,
 "ValidatedLazyProfile.reputation expected <class 'int'>, got str (not a number)")

`Columns()` turns a dict-of-lists payload into `list[RowModel]`. `Alias(...)` still works inside the row model, so API-shaped column names can map to Pythonic attributes. For keys with punctuation, use a quoted raw key like `Alias('"sku_#"')`.

In [49]:
@apimodel
class InventoryRow:
    sku: Annotated[str, Alias('"sku_#"')]
    name: str
    qty: int

@apimodel
class InventorySnapshot:
    rows: Annotated[list[InventoryRow], Columns(required=("sku_#", "qty"))]
    
snapshot = InventorySnapshot({
    "rows": {
        "sku_#": ["A-100", "B-205", "C-310"],
        "name": ["Cable", "Battery", "Adapter"],
        "qty": [3, 12, 1],
    },
})

assert len(snapshot.rows) == 3
assert snapshot.rows[0].sku == "A-100"
assert snapshot.rows[1].name == "Battery"
assert snapshot.rows[2].qty == 1

for row in snapshot.rows:
    print(row)

InventoryRow(sku='A-100', name='Cable', qty=3)
InventoryRow(sku='B-205', name='Battery', qty=12)
InventoryRow(sku='C-310', name='Adapter', qty=1)


`Timestamp()` parses raw API timestamp values into `datetime`.

In [50]:
@apimodel(validate=True)
class JobRun:
    started_at: Annotated[datetime, Timestamp()]

run = JobRun({"started_at": "2026-04-19T12:00:00Z"})

assert run.started_at == datetime(2026, 4, 19, 12, 0, tzinfo=timezone.utc)

{
    "started_at_type": type(run.started_at).__name__,
    "started_at": run.started_at.isoformat(),
}

{'started_at_type': 'datetime', 'started_at': '2026-04-19T12:00:00+00:00'}

## Nested Models

`instantiate_type` understands `list[T]`, `dict[K, V]`, dataclasses, and classes with `from_dict`. Because `@apimodel` adds `from_dict`, nested API models compose naturally.

In [51]:
@apimodel
class Address:
    city: str
    region: str
    country: str

@apimodel
class Profile:
    email: str
    timezone: str
    address: Address

@apimodel
class Person:
    id: int
    name: str
    profile: Profile

@apimodel
class Artifact:
    sha256: str
    bytes: int

@apimodel
class Release:
    version: str
    notes: list[str]
    artifacts: dict[str, Artifact]

@apimodel
class Latency:
    p50: int
    p95: int
    p99: int

@apimodel
class Metrics:
    latency_ms: Latency
    daily_requests: list[int]

@apimodel
class Project:
    id: int
    name: str
    status: str
    releases: list[Release]
    metrics: Metrics

@apimodel
class AuditEvent:
    at: str
    kind: str
    actor: str
    details: dict[str, object]

@apimodel
class AuditTrail:
    created_by: Person
    events: list[AuditEvent]

@apimodel
class Stats:
    open_issues: int
    stars: int
    labels: list[str]

## Lazy Fields

Fields annotated as `Lazy[T]` are not hydrated in `__init__`. The decorator installs descriptors for them, so hydration happens only when the attribute is first read.

In [52]:
from typing import List


@apimodel
class Organization:
    id: int
    slug: str
    active: bool
    owner: Lazy[Person]
    projects: Lazy[List[Project]]
    audit: Lazy[Annotated[AuditTrail, Alias("auditTrail")]]
    stats: Lazy[Stats]

@apimodel
class Page:
    number: int
    size: int
    next: str

@apimodel
class ApiEnvelope:
    request_id: str
    generated_at: str
    page: Page
    organizations: List[Organization]

Load the moderately nested fixture and hydrate the envelope. The list of organizations is eager, but each organization's heavy branches are still lazy.

In [53]:
raw = load_json(data_path)
envelope = ApiEnvelope(raw)
first_org = envelope.organizations[0]

assert envelope.page.size == 3
assert len(envelope.organizations) == 3
assert isinstance(first_org, Organization)
assert lazy_cache_keys(first_org) == ()

first_org.slug, lazy_cache_keys(first_org)

('atlas-foundry', ())

Accessing a lazy branch hydrates only that branch. The owner has nested eager models inside it, but projects and audit are untouched.

In [54]:
owner = first_org.owner

assert isinstance(owner, Person)
assert owner.profile.address.city == "Flagstaff"
assert lazy_cache_keys(first_org) == ("_lazycache_owner",)

owner.name, owner.profile.timezone, lazy_cache_keys(first_org)

('Mina Vale', 'America/Phoenix', ('_lazycache_owner',))

A lazy list can still hydrate into deeply nested typed models. This is the main cost-saving shape for large API payloads: keep the top-level index cheap, and pay for deep structures only when you need them.

In [55]:
projects = first_org.projects
latest = projects[0].releases[0]

assert isinstance(projects[0], Project)
assert isinstance(latest.artifacts["wheel"], Artifact)
assert latest.artifacts["wheel"].bytes == 182144
assert "_lazycache_projects" in lazy_cache_keys(first_org)

[project_line(project) for project in projects]

['9001: Cinder Index (green)', '9002: Needle Cache (yellow)']

## Aliases

`Alias("raw_key")` lives inside `typing.Annotated`. The model keeps Python-friendly attribute names while reading from API-shaped keys. When an alias is present, `@apimodel` looks only at that raw key; a same-named Python field in the payload is ignored.


In [56]:
@apimodel
class AliasProfile:
    display_name: Annotated[str, Alias("displayName")]
    request_count: Annotated[int, Alias("requestCount")]

profile = AliasProfile({
    "displayName": "Mina Vale",
    "display_name": "Wrong key",
    "requestCount": "42",
})
field_name_only = AliasProfile({
    "display_name": "Ignored without displayName",
    "requestCount": 7,
})

assert profile.display_name == "Mina Vale"
assert profile.request_count == 42
assert field_name_only.display_name is UNSET

@apimodel
class AliasedLazyOwner:
    owner: Lazy[Annotated[Person, Alias("ownerProfile")]]

lazy_owner = AliasedLazyOwner({"ownerProfile": raw["organizations"][0]["owner"]})
assert lazy_cache_keys(lazy_owner) == ()
assert lazy_owner.owner.name == "Mina Vale"
assert lazy_cache_keys(lazy_owner) == ("_lazycache_owner",)

# The larger fixture uses the same lazy alias mechanism: Organization.audit reads auditTrail.
audit = first_org.audit
assert isinstance(audit, AuditTrail)
assert audit.created_by.profile.address.region == "CO"
assert [event.kind for event in audit.events] == ["create", "sync"]

{
    "eager_alias": profile.display_name,
    "field_name_only": repr(field_name_only.display_name),
    "lazy_alias": lazy_owner.owner.profile.email,
    "fixture_alias_events": [event.details for event in audit.events],
}


{'eager_alias': 'Mina Vale',
 'field_name_only': 'UNSET',
 'lazy_alias': 'mina@example.test',
 'fixture_alias_events': [{'source': 'import', 'rows': 42},
  {'source': 'api', 'rows': 11}]}

## Timestamp Parsing

`Timestamp(...)` is another `typing.Annotated` config object. It lets the model expose a field as `datetime` even when the API sends an ISO string, Unix timestamp, millisecond timestamp, or a custom formatted string. It also composes with `Alias(...)` and `Lazy[...]`.

In [57]:
@apimodel(validate=True)
class TimestampedEvent:
    created_at: Annotated[datetime, Alias("createdAt"), Timestamp()]
    synced_at: Annotated[datetime, Timestamp(unit="ms")]
    reviewed_at: Annotated[datetime, Timestamp("%Y/%m/%d %H:%M:%S")]

event = TimestampedEvent({
    "createdAt": "2026-04-19T12:34:56Z",
    "synced_at": 1776602096000,
    "reviewed_at": "2026/04/19 12:35:12",
})

assert isinstance(event.created_at, datetime)
assert event.created_at == datetime(2026, 4, 19, 12, 34, 56, tzinfo=timezone.utc)
assert event.synced_at == datetime(2026, 4, 19, 12, 34, 56, tzinfo=timezone.utc)
assert event.reviewed_at == datetime(2026, 4, 19, 12, 35, 12, tzinfo=timezone.utc)

{
    "created_at_type": type(event.created_at).__name__,
    "created_at": event.created_at.isoformat(),
    "synced_at": event.synced_at.isoformat(),
    "reviewed_at": event.reviewed_at.isoformat(),
}


{'created_at_type': 'datetime',
 'created_at': '2026-04-19T12:34:56+00:00',
 'synced_at': '2026-04-19T12:34:56+00:00',
 'reviewed_at': '2026-04-19T12:35:12+00:00'}

In [58]:
@apimodel(validate=True)
class LazyTimestampedAudit:
    first_seen: Lazy[Annotated[datetime, Alias("firstSeen"), Timestamp()]]
    last_seen: Lazy[Annotated[datetime, Timestamp(unit="ms")]]

lazy_audit = LazyTimestampedAudit({
    "firstSeen": "2026-04-18T10:15:00-07:00",
    "last_seen": 1776602096000,
})

# Validation forces lazy fields to hydrate during construction, so the cache keys are already present.
assert lazy_cache_keys(lazy_audit) == ("_lazycache_first_seen", "_lazycache_last_seen")
assert isinstance(lazy_audit.first_seen, datetime)
assert lazy_audit.last_seen == datetime(2026, 4, 19, 12, 34, 56, tzinfo=timezone.utc)

try:
    TimestampedEvent({
        "createdAt": "not a timestamp",
        "synced_at": 1776602096000,
        "reviewed_at": "2026/04/19 12:35:12",
    })
except ApiModelValidationError as exc:
    validation_message = str(exc)

{
    "first_seen": lazy_audit.first_seen.isoformat(),
    "last_seen": lazy_audit.last_seen.isoformat(),
    "invalid_payload": validation_message,
}


{'first_seen': '2026-04-18T10:15:00-07:00',
 'last_seen': '2026-04-19T12:34:56+00:00',
 'invalid_payload': "TimestampedEvent.created_at expected <class 'datetime.datetime'>, got str (not a timestamp)"}

## Cache Behavior

A lazy value is cached per instance. Repeated reads return the same object. Deleting the attribute clears that cached value, so the next read rehydrates from the original raw dict.

In [59]:
again = first_org.projects
assert again is projects

raw["organizations"][0]["projects"][0]["status"] = "blue"
assert first_org.projects[0].status == "green"

del first_org.projects
rehydrated_projects = first_org.projects

assert rehydrated_projects is not projects
assert rehydrated_projects[0].status == "blue"

project_line(rehydrated_projects[0])

'9001: Cinder Index (blue)'

## Independent Instances

The descriptor lives on the class, but cached lazy values live on each instance. Hydrating one organization does not hydrate another.

In [60]:
second_org = envelope.organizations[1]
third_org = envelope.organizations[2]

assert lazy_cache_keys(second_org) == ()
assert second_org.owner.name == "Jo Rowan"
assert lazy_cache_keys(second_org) == ("_lazycache_owner",)
assert lazy_cache_keys(third_org) == ()

second_org.owner.profile.address.city, lazy_cache_keys(third_org)

('Rochester', ())

## Round Trips and Equality

`from_dict` mirrors direct construction, and equality compares `as_dict()` output for instances of the same class.

In [61]:
raw_again = copy.deepcopy(load_json(data_path))
left = ApiEnvelope.from_dict(raw_again)
right = ApiEnvelope(raw_again)

assert left == right
assert left.as_dict() is raw_again
assert repr(left.organizations[0]).startswith("Organization(")

left == right, repr(left.organizations[0])

(True,
 "Organization(id=101, slug='atlas-foundry', active=True, owner=Person(id=5001, name='Mina Vale', profile=Profile(email='mina@example.test', timezone='America/Phoenix', address=Address(city='Flagstaff', region='AZ', country='US'))), projects=[Project(id=9001, name='Cinder Index', status='green', releases=[Release(version='1.2.0', notes=['metadata paging', 'faster cold reads'], artifacts={'wheel': Artifact(sha256='demo-wheel-9001-120', bytes=182144), 'sdist': Artifact(sha256='demo-sdist-9001-120', bytes=74211)}), Release(version='1.1.4', notes=['retry tagging', 'trace cleanup'], artifacts={'wheel': Artifact(sha256='demo-wheel-9001-114', bytes=175002)})], metrics=Metrics(latency_ms=Latency(p50=18, p95=44, p99=79), daily_requests=[820, 901, 977, 1044, 998, 1120, 1186])), Project(id=9002, name='Needle Cache', status='yellow', releases=[Release(version='0.9.3', notes=['sample compaction', 'manifest repair'], artifacts={'wheel': Artifact(sha256='demo-wheel-9002-093', bytes=160313)})],

## Things Worth Noticing

- `as_dict()` returns the original raw dict, not a copy.
- Eager fields hydrate during construction; lazy fields hydrate on first read.
- Lazy caches are stored on the instance under `_lazycache_<field>`.
- `del obj.lazy_field` clears one lazy cache and forces the next access to re-read raw data.
- `Alias(...)` maps model names to API keys for eager and lazy fields.
- Lazy fields are deferred shaping, not refreshable living objects.